**Objective**

Clean the combined master_metadata.csv and produce a standardized dataset for EDA.

Input: data/processed/master_metadata.csv

Output:
data/processed/cleaned_metadata.csv
reports/data_cleaning_log.csv

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
PROJECT_DIR = Path("/workspaces/DBA-ai-trust-score-framework-Interim")

PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
REPORT_DIR = PROJECT_DIR / "reports"

REPORT_DIR.mkdir(exist_ok=True)

In [3]:
metadata = pd.read_csv(
    PROCESSED_DIR / "master_metadata.csv"
)

metadata.head()

,age,gender,race,service_test,file_name,dataset,Image_id,Left_eye_react,Right_eye_react,Label
0,4.0,0.0,3.0,False,fairface_00000.jpg,FairFace,NaN,NaN,NaN,NaN
1,1.0,1.0,5.0,False,fairface_00001.jpg,FairFace,NaN,NaN,NaN,NaN
2,2.0,1.0,0.0,False,fairface_00002.jpg,FairFace,NaN,NaN,NaN,NaN
3,3.0,1.0,0.0,False,fairface_00003.jpg,FairFace,NaN,NaN,NaN,NaN
4,2.0,0.0,5.0,True,fairface_00004.jpg,FairFace,NaN,NaN,NaN,NaN


In [4]:
print("Shape :", metadata.shape)

print("\nColumns:")
print(metadata.columns.tolist())

print("\nData Types:")
print(metadata.dtypes)

Shape : (2400, 10)

Columns:
['age', 'gender', 'race', 'service_test', 'file_name', 'dataset', 'Image_id', 'Left_eye_react', 'Right_eye_react', 'Label']

Data Types:
age                float64
gender             float64
race               float64
service_test        object
file_name              str
dataset                str
Image_id           float64
Left_eye_react         str
Right_eye_react        str
Label                  str
dtype: object


In [5]:
missing = metadata.isnull().sum()

missing = missing[
    missing > 0
].sort_values(
    ascending=False
)

missing

age                1400
gender             1400
race               1400
service_test       1400
Image_id           1400
Left_eye_react     1400
Right_eye_react    1400
Label              1400
dtype: int64

________________Duplicate Records_________________

In [6]:
duplicates = metadata.duplicated().sum()

print("Duplicate records:", duplicates)

Duplicate records: 0


_________________Remove Duplicates____________________

In [7]:
metadata = metadata.drop_duplicates()

print(metadata.shape)

(2400, 10)


__________________Standardize column names_________________

In [8]:
metadata.columns = (
    metadata.columns
    .str.lower()
    .str.strip()
    .str.replace(" ", "_")
)

metadata.columns

Index(['age', 'gender', 'race', 'service_test', 'file_name', 'dataset',
       'image_id', 'left_eye_react', 'right_eye_react', 'label'],
      dtype='str')

______________________Removing Empty cells____________________

In [9]:
metadata = metadata.dropna(
    axis=1,
    how="all"
)

metadata.shape

(2400, 10)

________________________Fill Missing Cells________________________

In [10]:
import pandas as pd

for col in metadata.columns:

    # Numeric columns
    if pd.api.types.is_numeric_dtype(metadata[col]):

        metadata[col] = metadata[col].fillna(
            metadata[col].median()
        )

    # Everything else
    else:

        metadata[col] = metadata[col].fillna("Unknown")

__________________________Data Cleaning Log________________________

In [11]:
cleaning_log = pd.DataFrame({

    "Issue":[

        "Missing Values",
        "Duplicate Records",
        "Columns Removed"

    ],

    "Count":[

        missing.sum(),
        duplicates,
        0

    ],

    "Treatment":[

        "Median/Unknown Imputation",
        "Removed",
        "None"

    ]

})

cleaning_log

,Issue,Count,Treatment
0,Missing Values,11200,Median/Unknown Imputation
1,Duplicate Records,0,Removed
2,Columns Removed,0,None


__________________________Save Outputs_______________________

In [12]:
metadata.to_csv(
    PROCESSED_DIR / "cleaned_metadata.csv",
    index=False
)

cleaning_log.to_csv(
    REPORT_DIR / "data_cleaning_log.csv",
    index=False
)

print("✅ cleaned_metadata.csv saved")
print("✅ data_cleaning_log.csv saved")

✅ cleaned_metadata.csv saved
✅ data_cleaning_log.csv saved


___________________________Final Validation________________________

In [13]:
print("Final Shape:", metadata.shape)

print("\nMissing Values Remaining")

print(
    metadata.isnull().sum().sum()
)

print("\nDuplicate Records")

print(
    metadata.duplicated().sum()
)

Final Shape: (2400, 10)

Missing Values Remaining
0

Duplicate Records
0
